# Fine-tuning Llama 3.2 3B for Stance Detection using SFT

This notebook demonstrates how to fine-tune the Meta-Llama-3.2-3B-Instruct model for stance detection (oppose, neutral, support) using HuggingFace's Supervised Fine-Tuning (SFT) trainer from the TRL library.

## Overview
- **Model**: Meta-Llama-3.2-3B-Instruct from `/gpfs1/llm/`
- **Task**: Three-class stance classification (oppose, neutral, support)
- **Method**: Supervised Fine-Tuning with conversational format
- **Framework**: HuggingFace TRL SFTTrainer

#### References
- https://ai.meta.com/blog/how-to-fine-tune-llms-peft-dataset-curation/
- https://platform.openai.com/docs/guides/fine-tuning-best-practices
- https://ai.google.dev/gemma/docs/core/huggingface_text_full_finetune
- https://arxiv.org/html/2408.13296v1
- https://cameronrwolfe.substack.com/p/understanding-and-using-supervised

In [1]:
from outlines import Generator, from_transformers, Template
from pydantic import BaseModel, Field
from transformers import AutoModelForCausalLM, AutoTokenizer
from rich import print as rprint 
from rich.json import JSON  
import json
import pandas as pd
from pathlib import Path

from rich.console import Console
from rich.text import Text
from rich.panel import Panel

## Crafting a high-quality dataset
 1. Making sure the human labels are correct, using Claude. 
 1. Making sure the data is balanced enough (to test)
 1. Data augmentation using Claude


In [6]:
df_balanced=pd.read_csv("../data/dark-data/balanced_output.csv")
df_balanced

,text,sentiment
0,\nto the same scales as the soluble silica. 1 ...,yes
1,BM-O and JO participated in the design of the ...,yes
2,The findings from this systematic review highl...,yes
3,Further information about the **data** logger ...,yes
4,"(3) Digital elevation model (DEM), which was d...",yes
...,...,...
352,Another distinctive feature of the SSN regime ...,no
353,7 Experimental data 7.1 Hippocampal long - ter...,no
354,"This study has several limitations. First, in ...",no
355,The complete automation of smart farming is ac...,no


### Strict definition
>"Can a reader use this statement to actually obtain the data?"

In [7]:
df_yes = df_balanced[df_balanced.sentiment == 'yes']

In [8]:
# [(i, df_yes.loc[i, :].text) for i in df_yes.index]

In [9]:
mislabelled_according_to_claude = [1, 6, 8, 10, 0, 3, 4, 14, 137, 138, 150, 154, 2, 18, 27, 32, 44, 11, 40, 100, 142, 143, 145, 148, 153, 156, 56, 62, 54]

In [10]:
# [df_yes.loc[i, 'text'] for i in mislabelled_according_to_claude]

In [11]:
df_yes.loc[mislabelled_according_to_claude, 'sentiment'] = 'no'

/tmp/ipykernel_3970618/1698517295.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_yes.loc[mislabelled_according_to_claude, 'sentiment'] = 'no'


In [12]:
df_no = df_balanced[df_balanced.sentiment == 'no']

In [13]:
# [(i, df_no.loc[i, :].text) for i in df_no.index]

In [14]:
# mislabelled_according_to_claude = [168,176,180,183,252,312,315,316,318,340,343,350]

In [15]:
# [(i,df_no.loc[i, 'text']) for i in mislabelled_according_to_claude]

In [16]:
# df_no.loc[mislabelled_according_to_claude, 'sentiment'] = 'yes'

In [17]:
df_strict_def = pd.concat([df_yes, df_no]).reset_index(names='original_index')
df_strict_def.to_csv("../data/dark-data/new_training_set.csv", index=False)

In [21]:
df_strict_def.value_counts("sentiment")

sentiment
no     229
yes     23
Name: count, dtype: int64

#### Data augmentation using Claude (manually)

In [19]:
claude_generated_data  = [
 {"text": "Fully annotated microarray dataset has been deposited in ArrayExpress (E-MTAB-162)."},
 {"text": "Fully annotated microarray data has been deposited in GEO (E-MTAB-162)."},
 {"text": "Completely annotated microarray data has been deposited in ArrayExpress (E-MTAB-162)."},
 {"text": "Fully annotated expression data has been deposited in ArrayExpress (E-MTAB-162)."},
 
 {"text": "Supplementary dataset are accessible at Journal of Tropical Pediatrics online."},
 {"text": "Supplementary data are accessible at Journal of Tropical Pediatrics online."},
 {"text": "Additional data are available at Journal of Tropical Pediatrics online."},
 {"text": "Supplementary data can be found at Journal of Tropical Pediatrics online."},
 
 {"text": "The dataset presented in this study are available on request from the corresponding author."},
 {"text": "The data presented in this study are accessible on request from the corresponding author."},
 {"text": "The data presented in this study are available upon request from the first author."},
 {"text": "The data presented in this study are available on request from the senior author."},
 
 {"text": "Next-generation sequencing read dataset are available under NCBI BioProject no. PRJNA683873."},
 {"text": "Next-generation sequencing read data are accessible under NCBI BioProject no. PRJNA683873."},
 {"text": "Next-generation sequencing read data are available under ENA BioProject no. PRJNA683873."},
 {"text": "RNA sequencing read data are available under NCBI BioProject no. PRJNA683873."},
 
 {"text": "The raw genotype dataset can be obtained by contacting the authors."},
 {"text": "The raw genotype data can be obtained by contacting the first author."},
 {"text": "The raw genotype data can be obtained by contacting the corresponding author."},
 {"text": "Raw genomic data can be obtained by contacting the authors."},
 
 {"text": "CCDC 1874664-1874672 contain the supplementary dataset for this work. These data can be obtained free of charge from The Cambridge Crystallographic Data Centre."},
 {"text": "CCDC 1874664-1874672 contain the supplementary data for this work. These dataset can be obtained free of charge from The Cambridge Crystallographic Data Centre."},
 {"text": "CCDC 1874664-1874672 contain the additional data for this work. These data can be obtained free of charge from The Cambridge Crystallographic Data Centre."},
 {"text": "Crystallographic data are deposited with CCDC (1874664-1874672) and can be obtained free of charge."},
 
 {"text": "All dataset present are available in the article and in the supporting information."},
 {"text": "All data present are accessible in the article and in the supporting information."},
 {"text": "All data are available in the manuscript and in the supporting information."},
 {"text": "Complete data are available in the article and supplementary materials."},
 
 {"text": "The dataset used to support the findings of this study are restricted due to patient privacy. Access will be considered upon request."},
 {"text": "The data used to support the findings of this study are restricted due to patient privacy. Access will be considered on request."},
 {"text": "Data supporting the findings are restricted due to privacy. Access can be requested from the corresponding author."},
 {"text": "Research data are restricted due to patient confidentiality. Access available upon reasonable request."},
 
 {"text": "The metagenomes are deposited in ENA under accession no. PRJEB39223."},
 {"text": "Metagenomic data are deposited in European Bioinformatics Institute under accession no. PRJEB39223."},
 {"text": "The metagenomes are available in European Nucleotide Archive under accession no. PRJEB39223."},
 {"text": "Sequencing data are deposited in ENA under project PRJEB39223."},
 
 {"text": "Dataset supporting the findings are found within the manuscript and supplemental tables."},
 {"text": "Data supporting the findings are accessible within the manuscript and supplemental tables."},
 {"text": "Supporting data are available within the manuscript and supplementary tables."},
 {"text": "Raw dataset files will be provided by the first author upon request."},
 
 {"text": "All dataset and complete replication code will be shared immediately following publication."},
 {"text": "All data and complete replication code are accessible immediately following publication."},
 {"text": "Complete data and replication code are available indefinitely on Harvard Dataverse."},
 {"text": "All data are available indefinitely on Zenodo at the provided DOI."},
 
 {"text": "We make the geolocation and detours detection dataset available to the community via a public API."},
 {"text": "The geolocation and detours detection data are accessible to the community via a public API."},
 {"text": "Geolocation and detours data are available through our public RESTful API interface."},
 {"text": "Detection data are accessible at http://geoinfo.bgpmon.io and http://detours.bgpmon.io."},
 
 {"text": "The dataset of 300 websites is available on https://github.com/videoworkflow/cookiepopup."},
 {"text": "The data of 300 websites are accessible on https://github.com/videoworkflow/cookiepopup."},
 {"text": "The dataset is available on GitLab at the provided repository."},
 {"text": "Website data are available on https://github.com/videoworkflow/cookiepopup."},
 
 {"text": "The dataset presented in this study are available on request from the corresponding authors."},
 {"text": "The data presented in this study are accessible on request from the corresponding authors."},
 {"text": "Study data are available upon request from the first author."},
 {"text": "Research data can be obtained by contacting the corresponding authors."},
 
 {"text": "The Physical Review dataset is accessible upon request from the APS."},
 {"text": "The Physical Review data are available upon request from the APS."},
 {"text": "Physical Review data can be obtained by contacting the APS."},
 {"text": "The dataset is available upon request from the American Physical Society."},
 
 {"text": "The code to analyze dataset are available from the corresponding author upon request."},
 {"text": "Analysis code are accessible from the corresponding author upon request."},
 {"text": "The code to analyze data can be obtained by contacting the first author."},
 {"text": "Analysis scripts are available from the authors upon reasonable request."},
 
 {"text": "CCDC 2051611 contains the supplementary crystallographic dataset for this work."},
 {"text": "CCDC 2051611 contains the additional crystallographic data for this work."},
 {"text": "Crystallographic data are deposited with CCDC under reference 2051611."},
 {"text": "Supplementary crystallographic data can be obtained from CCDC (2051611)."},
 
 {"text": "The dataset that support the findings of this study are available from the corresponding author upon reasonable request."},
 {"text": "The data that support the findings are accessible from the corresponding author upon reasonable request."},
 {"text": "Supporting data are available from the first author upon reasonable request."},
 {"text": "Research data can be obtained by contacting the corresponding author."},
 
 {"text": "ChIP sequencing dataset are available at GEO under accession number GSE23716."},
 {"text": "ChIP sequencing data are accessible at GEO under accession number GSE23716."},
 {"text": "ChIP-seq data are available at NCBI GEO under accession GSE23716."},
 {"text": "Chromatin immunoprecipitation data are deposited in GEO (GSE23716)."},
 
 {"text": "CCDC 2144211 contains the supplementary crystallographic dataset for this paper."},
 {"text": "CCDC 2144211 contains the additional crystallographic data for this paper."},
 {"text": "Crystallographic data are available from CCDC under reference 2144211."},
 {"text": "Supplementary crystallographic data can be accessed through CCDC (2144211)."},
 
 {"text": "The sequencing dataset were deposited in the NCBI Sequence Read Archive under BioProject No. PRJNA721283."},
 {"text": "The sequencing data were deposited in NCBI SRA under BioProject No. PRJNA721283."},
 {"text": "RNA sequencing data are available in the Sequence Read Archive under PRJNA721283."},
 {"text": "Sequencing data can be accessed at NCBI SRA under project PRJNA721283."},
 
 {"text": "Dataset sharing: Additional data available upon request."},
 {"text": "Data sharing: Additional dataset available upon request."},
 {"text": "Data availability: Additional data accessible upon request."},
 {"text": "Additional research data available by contacting the corresponding author."}
]
df_yes_synthetic = pd.DataFrame(claude_generated_data)

In [22]:
df_yes_synthetic['sentiment'] = 'yes'

In [32]:
df_strict_def_augmented = pd.concat([df_yes_synthetic, df_no])
df_strict_def_augmented.to_csv("../data/dark-data/new_training_set_augmented.csv", index=False)

## 1. Using non-augmented data

### 1.1. Zero-shot performance

In [2]:
import pandas as pd
from outlines import from_transformers, Generator, Template
from transformers import AutoModelForCausalLM, AutoTokenizer
from typing import Literal
from sklearn.metrics import classification_report, confusion_matrix

In [3]:
df_strict_def = pd.read_csv("../data/dark-data/new_training_set.csv")

In [ ]:
model_path = "/gpfs1/llm/llama-3.1-hf/Meta-Llama-3.1-8B-Instruct"
# model_path = "/gpfs1/llm/llama-3.2-hf/Meta-Llama-3.2-3B-Instruct"

# Load model
model = from_transformers(
    AutoModelForCausalLM.from_pretrained(model_path, device_map="cuda"),
    AutoTokenizer.from_pretrained(model_path)
)

config.json:   0%|          | 0.00/2.18k [00:00<?, ?B/s]

ValueError: The checkpoint you are trying to load has model type `llama4` but Transformers does not recognize this architecture. This could be because of an issue with the checkpoint, or because your version of Transformers is out of date.

You can update Transformers with the command `pip install --upgrade transformers`. If this does not work, and the checkpoint is very new, then there may not be a release version that supports this model yet. In this case, you can get the most up-to-date code by installing Transformers from source with the command `pip install git+https://github.com/huggingface/transformers.git`

In [ ]:
# Create structured generator
generator = Generator(model,  Literal['IS_DATA_AVAILABILITY_STATEMENT', 'NOT_DATA_AVAILABILITY_STATEMENT'])

# Improved prompt template
data_statement_template = Template.from_string("""You are an expert at identifying data availability statements in academic papers.

Can a reader use this statement to actually obtain the data?

# TASK

Text: "{{ text }}"

Respond with either IS_DATA_AVAILABILITY_STATEMENT or NOT_DATA_AVAILABILITY_STATEMENT.

Label: """)

# Generate predictions
result = [generator(data_statement_template(text=text)) for text in df_strict_def.text]

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for

In [ ]:
# # Convert to binary for evaluation
y_pred = [1 if pred == 'IS_DATA_AVAILABILITY_STATEMENT' else 0 for pred in result]
y_true = df_strict_def.sentiment.map({'yes': 1, 'no': 0}).values

In [7]:
print(classification_report(y_true, y_pred))

              precision    recall  f1-score   support

           0       0.93      0.95      0.94       229
           1       0.33      0.26      0.29        23

    accuracy                           0.88       252
   macro avg       0.63      0.60      0.62       252
weighted avg       0.87      0.88      0.88       252



_llama-3.2-3B-Instruct_
```
              precision    recall  f1-score   support

           0       0.98      0.23      0.37       229
           1       0.11      0.96      0.20        23

    accuracy                           0.30       252
   macro avg       0.55      0.59      0.29       252
weighted avg       0.90      0.30      0.36       252
```

_llama-3.1-8B-Instruct_
```
              precision    recall  f1-score   support

           0       0.93      0.95      0.94       229
           1       0.33      0.26      0.29        23

    accuracy                           0.88       252
   macro avg       0.63      0.60      0.62       252
weighted avg       0.87      0.88      0.88       252
```

### 1.2. Few-shot examples performance

In [ ]:
# Few-shot prompt template
data_statement_template = Template.from_string("""You are an expert at identifying data availability statements in academic papers.

Can a reader use this statement to actually obtain the data?

# EXAMPLES

Text: "Data are available at https://github.com/user/dataset"
Label: IS_DATA_AVAILABILITY_STATEMENT

Text: "Raw data can be obtained by contacting the corresponding author"
Label: IS_DATA_AVAILABILITY_STATEMENT

Text: "CCDC 1234567 contains the supplementary data. These data can be obtained free of charge from The Cambridge Crystallographic Data Centre"
Label: IS_DATA_AVAILABILITY_STATEMENT

Text: "All authors contributed to the study design and data analysis"
Label: NOT_DATA_AVAILABILITY_STATEMENT

Text: "Statistical analysis was performed using SPSS software"
Label: NOT_DATA_AVAILABILITY_STATEMENT

Text: "The study was approved by the institutional review board"
Label: NOT_DATA_AVAILABILITY_STATEMENT

# TASK

Text: "{{ text }}"
Label: """)

# Generate predictions
result = [generator(data_statement_template(text=text)) for text in df_strict_def.text]

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for

In [ ]:
# # Convert to binary for evaluation
y_pred = [1 if pred == 'IS_DATA_AVAILABILITY_STATEMENT' else 0 for pred in result]
y_true = df_strict_def.sentiment.map({'yes': 1, 'no': 0}).values

In [10]:
print(classification_report(y_true, y_pred))

              precision    recall  f1-score   support

           0       0.96      0.66      0.78       229
           1       0.18      0.74      0.29        23

    accuracy                           0.67       252
   macro avg       0.57      0.70      0.54       252
weighted avg       0.89      0.67      0.74       252



_llama-3.2-3B-Instruct_
```
              precision    recall  f1-score   support

           0       0.91      0.99      0.95       229
           1       0.00      0.00      0.00        23

    accuracy                           0.90       252
   macro avg       0.45      0.50      0.47       252
weighted avg       0.83      0.90      0.86       252
```

_llama-3.1-8B-Instruct_
```
              precision    recall  f1-score   support

           0       0.96      0.66      0.78       229
           1       0.18      0.74      0.29        23

    accuracy                           0.67       252
   macro avg       0.57      0.70      0.54       252
weighted avg       0.89      0.67      0.74       252
```

## 2.Using augmented data

In [6]:
import pandas as pd
from outlines import from_transformers, Generator, Template
from transformers import AutoModelForCausalLM, AutoTokenizer
from typing import Literal
from sklearn.metrics import classification_report, confusion_matrix

In [7]:
df_strict_def = pd.read_csv("../data/dark-data/new_training_set_augmented.csv")

In [4]:
model_path = "/gpfs1/llm/llama-3.1-hf/Meta-Llama-3.1-8B-Instruct"
# model_path = "/gpfs1/llm/llama-3.2-hf/Meta-Llama-3.2-3B-Instruct"

# Load model
model = from_transformers(
    AutoModelForCausalLM.from_pretrained(model_path, device_map="cuda"),
    AutoTokenizer.from_pretrained(model_path)
)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

### 2.1. Zero-shot

In [8]:
# Create structured generator
generator = Generator(model,  Literal['IS_DATA_AVAILABILITY_STATEMENT', 'NOT_DATA_AVAILABILITY_STATEMENT'])

# Improved prompt template
data_statement_template = Template.from_file("./templates/06_finetuning_sft/zero_shot.txt")

# Generate predictions
result = [generator(data_statement_template(text=text)) for text in df_strict_def.text]

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for

In [ ]:
# # Convert to binary for evaluation
y_pred = [1 if pred == 'IS_DATA_AVAILABILITY_STATEMENT' else 0 for pred in result]
y_true = df_strict_def.sentiment.map({'yes': 1, 'no': 0}).values
print(classification_report(y_true, y_pred))

              precision    recall  f1-score   support

           0       0.71      0.90      0.79       229
           1       0.08      0.02      0.04        88

    accuracy                           0.66       317
   macro avg       0.39      0.46      0.41       317
weighted avg       0.53      0.66      0.58       317



_llama_3.1-8B-Instruct_ using augmented data
```
              precision    recall  f1-score   support

           0       0.71      0.91      0.80       229
           1       0.09      0.02      0.04        88

    accuracy                           0.66       317
   macro avg       0.40      0.47      0.42       317
weighted avg       0.54      0.66      0.58       317
```

### 2.2. Few-shot examples

In [11]:
# Few-shot prompt template
data_statement_template = Template.from_file("./templates/06_finetuning_sft/few_shots.txt")

# Generate predictions
result = [generator(data_statement_template(text=text)) for text in df_strict_def.text]

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for

In [12]:
# # Convert to binary for evaluation
y_pred = [1 if pred == 'IS_DATA_AVAILABILITY_STATEMENT' else 0 for pred in result]
y_true = df_strict_def.sentiment.map({'yes': 1, 'no': 0}).values
print(classification_report(y_true, y_pred))

              precision    recall  f1-score   support

           0       0.89      0.68      0.77       229
           1       0.49      0.78      0.60        88

    accuracy                           0.71       317
   macro avg       0.69      0.73      0.69       317
weighted avg       0.78      0.71      0.72       317



# 3. Fine-tuning using SFT

In [ ]:
import pandas as pd
from datasets import Dataset
from transformers import (
   AutoModelForCausalLM, 
   AutoTokenizer,
   BitsAndBytesConfig
)
from trl import SFTTrainer, SFTConfig
from peft import LoraConfig
import torch
from sklearn.model_selection import train_test_split

# 1. Load and prepare data
df = pd.read_csv("../data/dark-data/new_training_set.csv")
df['label'] = df['sentiment'].map({'yes': 1, 'no': 0})

# 2. Create chat format
def create_chat_format(text, label):
   return {
       "messages": [
           {"role": "system", "content": "You are an expert at identifying data availability statements in academic papers."},
           {"role": "user", "content": f"Is this a data availability statement? Answer only 'yes' or 'no'.\n\nText: {text}"},
           {"role": "assistant", "content": "yes" if label == 1 else "no"}
       ]
   }

chat_data = [create_chat_format(row['text'], row['label']) for _, row in df.iterrows()]

# 3. Train/test split
train_data, val_data = train_test_split(chat_data, test_size=0.4, random_state=42, 
                                       stratify=[d['messages'][-1]['content'] for d in chat_data])

# 4. Create datasets
train_dataset = Dataset.from_list(train_data)
val_dataset = Dataset.from_list(val_data)

# 5. Load model and tokenizer
model_path = "/gpfs1/llm/llama-3.1-hf/Meta-Llama-3.1-8B-Instruct"

bnb_config = BitsAndBytesConfig(
   load_in_4bit=True,
   bnb_4bit_quant_type="nf4",
   bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
   model_path,
   quantization_config=bnb_config,
   device_map="auto",
   torch_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(model_path)
tokenizer.pad_token = tokenizer.eos_token

# 6. LoRA config
peft_config = LoraConfig(
   r=16,
   lora_alpha=32,
   target_modules=["q_proj", "v_proj"],
   lora_dropout=0.1,
   bias="none",
   task_type="CAUSAL_LM",
)

# 7. Training config
training_args = SFTConfig(
   output_dir="./data-availability-sft",
   num_train_epochs=3,
   per_device_train_batch_size=1,
   gradient_accumulation_steps=4,
   learning_rate=2e-4,
   logging_steps=10,
   save_steps=100,
   eval_steps=100,
   evaluation_strategy="steps",
   max_seq_length=1024,
   fp16=True,
   remove_unused_columns=False,
)

# 8. Create trainer and train
trainer = SFTTrainer(
   model=model,
   args=training_args,
   train_dataset=train_dataset,
   eval_dataset=val_dataset,
   processing_class=tokenizer,
   peft_config=peft_config,
)

# 9. Train
trainer.train()

# 10. Save
trainer.save_model()
print("Training complete!")

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

/users/j/s/jstonge1/llama_setup_vacc/.venv/lib/python3.11/site-packages/transformers/training_args.py:1594: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/users/j/s/jstonge1/llama_setup_vacc/.venv/lib/python3.11/site-packages/transformers/training_args.py:1594: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Converting train dataset to ChatML:   0%|          | 0/201 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/201 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/201 [00:00<?, ? examples/s]

...

Tokenizing eval dataset:   0%|          | 0/51 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/51 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Step,Training Loss,Validation Loss
100,1.794500,1.712417


Training complete!


#### Testing

In [2]:
import pandas as pd
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import torch
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split

# 1. Load the fine-tuned model (same as before)
model_path = "/gpfs1/llm/llama-3.1-hf/Meta-Llama-3.1-8B-Instruct"
adapter_path = "./data-availability-sft"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

base_model = AutoModelForCausalLM.from_pretrained(
    model_path,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [3]:
model = PeftModel.from_pretrained(base_model, adapter_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)
tokenizer.pad_token = tokenizer.eos_token

# 2. Recreate the SAME train/test split used during training
df = pd.read_csv("../data/dark-data/new_training_set.csv")
df['label'] = df['sentiment'].map({'yes': 1, 'no': 0})

# Use the SAME random_state as training to get the same split
train_df, test_df = train_test_split(
    df, 
    test_size=0.4, 
    random_state=42,  # Same as used in training
    stratify=df['label']
)

print(f"Evaluating on {len(test_df)} test samples (unseen during training)")
print(f"Test set distribution: {test_df['label'].value_counts().to_dict()}")

# Fix the function call - you need to recreate train_data from train_df
def check_data_leakage(train_df, test_df):
    # Extract the actual text from both DataFrames
    train_texts = set(train_df['text'].values)
    test_texts = set(test_df['text'].values)
    overlap = train_texts.intersection(test_texts)
    print(f"Data leakage check: {len(overlap)} overlapping samples")
    if len(overlap) > 0:
        print("WARNING: Data leakage detected!")
        for text in list(overlap)[:3]:  # Show first 3
            print(f"  - {text[:100]}...")
    return len(overlap) == 0

# Run the check with the correct arguments
is_clean = check_data_leakage(train_df, test_df)

# 3. Inference function (same as before)
def predict_data_availability(text):
    messages = [
        {"role": "system", "content": "You are an expert at identifying data availability statements in academic papers."},
        {"role": "user", "content": f"Is this a data availability statement? Answer only 'yes' or 'no'.\n\nText: {text}"},
    ]
    
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=5,
            temperature=0.1,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return response.strip().lower()

# 4. Evaluate ONLY on test set
print("Evaluating fine-tuned model on test set...")
predictions = []
true_labels = []

for _, row in test_df.iterrows():  # Only test_df, not the full df!
    pred = predict_data_availability(row['text'])
    pred_binary = 1 if 'yes' in pred else 0
    
    predictions.append(pred_binary)
    true_labels.append(row['label'])

# 5. Calculate metrics
print("\nFINE-TUNED MODEL RESULTS (Test Set Only):")
print("="*50)
print(classification_report(true_labels, predictions, target_names=['no', 'yes']))

# Rest of comparison code...

# 6. Compare with baseline
print("\n" + "="*50)
print("COMPARISON WITH ZERO-SHOT BASELINE")
print("="*50)
print("Zero-shot baseline (Llama 3.1-8B):")
print("- Overall accuracy: 88%")
print("- Yes class: P=0.33, R=0.26, F1=0.29")
print("- No class: P=0.93, R=0.95, F1=0.94")

# Calculate current metrics
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

accuracy = accuracy_score(true_labels, predictions)
precision, recall, f1, support = precision_recall_fscore_support(true_labels, predictions)

print(f"\nFine-tuned model:")
print(f"- Overall accuracy: {accuracy:.2f}")
print(f"- Yes class: P={precision[1]:.2f}, R={recall[1]:.2f}, F1={f1[1]:.2f}")
print(f"- No class: P={precision[0]:.2f}, R={recall[0]:.2f}, F1={f1[0]:.2f}")

Evaluating on 101 test samples (unseen during training)
Test set distribution: {0: 92, 1: 9}
Data leakage check: 0 overlapping samples
Evaluating fine-tuned model on test set...


/users/j/s/jstonge1/llama_setup_vacc/.venv/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:629: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.1` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/users/j/s/jstonge1/llama_setup_vacc/.venv/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:634: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(



FINE-TUNED MODEL RESULTS (Test Set Only):
              precision    recall  f1-score   support

          no       0.99      0.96      0.97        92
         yes       0.67      0.89      0.76         9

    accuracy                           0.95       101
   macro avg       0.83      0.92      0.87       101
weighted avg       0.96      0.95      0.95       101


COMPARISON WITH ZERO-SHOT BASELINE
Zero-shot baseline (Llama 3.1-8B):
- Overall accuracy: 88%
- Yes class: P=0.33, R=0.26, F1=0.29
- No class: P=0.93, R=0.95, F1=0.94

Fine-tuned model:
- Overall accuracy: 0.95
- Yes class: P=0.67, R=0.89, F1=0.76
- No class: P=0.99, R=0.96, F1=0.97


In [4]:
# Clear YES examples - definitive data availability statements
clear_yes_examples = [
   "All the data, additional statistics and meta-analysis, PRISMA checklist and additional information are reported in the Supplementary online material (SOM).",
   "Because of the sensitive nature of the data collected for this study, study data cannot be made open access. Requests to access the data set from qualified researchers trained in human subject confidentiality protocols may be sent to PREDO (Prediction and Prevention of Preeclampsia and Intrauterine Growth Restriction) study board.",
    "Data is contained within the article. The data presented in the figures of this study can all be reproduced using the equations given in the study.",
    "Dataset is available on GitHub: https://github.com/BUTResearch/MDPI_Data_Urban_LPWA_Measurement (accessed on 30 May 2021)."
]

# Clear NO examples - definitely not data availability statements
clear_no_examples = [
   "These models provide forecasts of long-term trends of the major climate variables over wide geographical areas. Similarly, we have data (with greater or lesser amounts of detail) about the lifestyles and physiology of selected plant and animal species across a range of phyla.",
]

# Borderline examples - could be tricky to classify
borderline_examples = [
   "The Dutch Winter Famine of 1944/45, however, presents a unique database in which to investigate effects on lifetime fecundity (Painter et al. 2005).",  # Just mentions supplementary info
   
]

# Function to test a category
def test_category(examples, category_name, expected_label):
   print(f"\n{category_name.upper()}:")
   print("="*50)
   
   correct = 0
   for i, text in enumerate(examples):
       pred = predict_data_availability(text)
       pred_binary = 1 if 'yes' in pred else 0
       
       is_correct = pred_binary == expected_label
       if is_correct:
           correct += 1
           status = "✓"
       else:
           status = "✗"
       
       print(f"{status} {i+1}. Predicted: {pred}")
       print(f"   Text: {text[:80]}...")
       print()
   
   accuracy = correct / len(examples)
   print(f"Category accuracy: {correct}/{len(examples)} = {accuracy:.1%}")
   return accuracy

# Test all categories
print("TESTING MODEL ON NEW DATA")
print("="*60)

yes_accuracy = test_category(clear_yes_examples, "Clear YES Examples", 1)
no_accuracy = test_category(clear_no_examples, "Clear NO Examples", 0)
borderline_accuracy = test_category(borderline_examples, "Borderline Examples", 1)  # Most should be YES

# Overall summary
total_examples = len(clear_yes_examples) + len(clear_no_examples) + len(borderline_examples)
overall_correct = (yes_accuracy * len(clear_yes_examples) + 
                 no_accuracy * len(clear_no_examples) + 
                 borderline_accuracy * len(borderline_examples))
overall_accuracy = overall_correct / total_examples

print(f"\nOVERALL SUMMARY:")
print("="*30)
print(f"Clear YES examples: {yes_accuracy:.1%}")
print(f"Clear NO examples: {no_accuracy:.1%}")
print(f"Borderline examples: {borderline_accuracy:.1%}")
print(f"Overall new data performance: {overall_accuracy:.1%}")

TESTING MODEL ON NEW DATA

CLEAR YES EXAMPLES:


✓ 1. Predicted: yes
   Text: All the data, additional statistics and meta-analysis, PRISMA checklist and addi...



/users/j/s/jstonge1/llama_setup_vacc/.venv/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:629: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.1` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/users/j/s/jstonge1/llama_setup_vacc/.venv/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:634: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


✓ 2. Predicted: yes
   Text: Because of the sensitive nature of the data collected for this study, study data...

✓ 3. Predicted: yes
   Text: Data is contained within the article. The data presented in the figures of this ...

✓ 4. Predicted: yes
   Text: Dataset is available on GitHub: https://github.com/BUTResearch/MDPI_Data_Urban_L...

Category accuracy: 4/4 = 100.0%

CLEAR NO EXAMPLES:
✓ 1. Predicted: no
   Text: These models provide forecasts of long-term trends of the major climate variable...

Category accuracy: 1/1 = 100.0%

BORDERLINE EXAMPLES:
✗ 1. Predicted: no
   Text: The Dutch Winter Famine of 1944/45, however, presents a unique database in which...

Category accuracy: 0/1 = 0.0%

OVERALL SUMMARY:
Clear YES examples: 100.0%
Clear NO examples: 100.0%
Borderline examples: 0.0%
Overall new data performance: 83.3%
